# Module 5: Feature Engineering for Child Welfare Data
**ACS Predictive Analytics Curriculum**

Concepts covered:
- Temporal/history features with dplyr window functions
- Allegation severity and escalation signals
- Reporter convergence features
- Child vulnerability interaction terms
- Compounding risk scores
- NLP features from case narratives (tidytext)
- Missingness as signal


In [ ]:
library(tidyverse)
library(tidytext)
library(slider)   # install if needed: install.packages('slider')

scr      <- read_csv('data/acs_scr_reports.csv', show_col_types=FALSE) %>%
  mutate(report_date = as.Date(report_date))
features <- read_csv('data/acs_features.csv', show_col_types=FALSE)

cat('Data loaded\n')

## SECTION 5.1: Temporal Features — History Over Time

In [ ]:
# RULE: Always sort by family_id + report_date before temporal features
scr_ordered <- scr %>%
  arrange(family_id, report_date)

temporal_features <- scr_ordered %>%
  group_by(family_id) %>%
  mutate(
    # Which report number is this for the family?
    report_number       = row_number(),

    # Total reports so far (not counting current)
    cumulative_reports  = row_number() - 1,

    # Days since last report (NA for first report)
    days_since_last     = as.numeric(report_date - lag(report_date)),

    # Is the gap SHRINKING? Escalation signal
    prev_gap            = lag(days_since_last),
    gap_shrinking       = days_since_last < prev_gap & !is.na(prev_gap),

    # Prior substantiations (not counting current)
    prior_subst_count   = cumsum(lag(outcome == 'Substantiated',
                                    default=FALSE)),

    # How many distinct reporter types have reported this family?
    n_prior_reporters   = cumsum(!duplicated(reporter_type)) - 1
  ) %>%
  ungroup()

temporal_features %>%
  select(report_id, family_id, report_date, report_number,
         days_since_last, gap_shrinking, prior_subst_count, n_prior_reporters) %>%
  head(15)

# DOMAIN INSIGHT: gap_shrinking is an escalation signal
# Family reported every 6 months, then 3 months, then 6 weeks
# Something is getting worse - prioritize for review

## SECTION 5.2: Allegation Severity and Escalation

In [ ]:
severity_map <- c(
  'Sexual Abuse'                         = 5,
  'Physical Abuse'                       = 4,
  'Domestic Violence Exposure'           = 3,
  'Neglect - Inadequate Supervision'     = 2,
  'Neglect - Medical'                    = 2,
  'Neglect - Inadequate Food/Clothing/Shelter' = 1,
  'Neglect - Educational'                = 1,
  'Emotional Abuse'                      = 1
)

severity_features <- temporal_features %>%
  mutate(
    allegation_severity = severity_map[allegation_type],
    allegation_severity = coalesce(allegation_severity, 1L)
  ) %>%
  group_by(family_id) %>%
  mutate(
    # Is this allegation more severe than the last?
    prev_severity          = lag(allegation_severity),
    severity_escalating    = allegation_severity > prev_severity & !is.na(prev_severity),

    # Max severity ever seen for this family
    max_prior_severity     = cummax(lag(allegation_severity, default=0)),

    # Has this specific allegation type appeared before?
    repeat_allegation_type = duplicated(allegation_type)
  ) %>%
  ungroup()

# Families with escalating severity - highest priority
severity_features %>%
  filter(severity_escalating) %>%
  select(family_id, report_date, allegation_type, allegation_severity,
         prev_severity, severity_escalating) %>%
  arrange(family_id, report_date) %>%
  head(10)

## SECTION 5.3: Reporter Convergence Features

In [ ]:
# DOMAIN INSIGHT: Multiple INDEPENDENT reporters = stronger signal
# One school reporting 5 times < 5 different institutions reporting once
# Independent convergence of concern is key

reporter_features <- severity_features %>%
  group_by(family_id) %>%
  mutate(
    # How many DISTINCT reporter types have flagged this family so far?
    distinct_reporters_so_far = cumsum(!duplicated(reporter_type)),

    # Is this report from a NEW type of reporter?
    new_reporter_type    = !duplicated(reporter_type),

    # Mandated reporter flag (legally required to report)
    is_mandated          = reporter_type %in%
      c('School/Educational Staff','Hospital/Medical','Daycare/Childcare'),

    # Anonymous reporter (lowest accuracy)
    is_anonymous         = reporter_type == 'Anonymous',

    # Medical reporter (highest accuracy - objective clinical observations)
    is_medical           = reporter_type == 'Hospital/Medical'
  ) %>%
  ungroup()

# Families reported by 3+ distinct reporter types = very high concern
reporter_features %>%
  group_by(family_id) %>%
  summarise(max_distinct_reporters = max(distinct_reporters_so_far)) %>%
  count(max_distinct_reporters) %>%
  mutate(pct = round(n/sum(n)*100,1))

# WHAT THIS MEANS:
# If 3 different institutions have independently flagged this family
# the likelihood of real maltreatment is much higher than
# if the same institution has flagged them 3 times

## SECTION 5.4: Child Vulnerability and Interaction Features

In [ ]:
vulnerability_features <- reporter_features %>%
  mutate(
    # Basic vulnerability flags
    child_age_under_2    = child_age < 2,   # Infants - highest vulnerability
    child_age_under_5    = child_age < 5,   # Young children

    # INTERACTION: Infant + Physical Abuse = highest risk combination
    # Risk is MULTIPLICATIVE not additive
    infant_physical_abuse = child_age < 2 & allegation_type == 'Physical Abuse',

    # INTERACTION: DV + Young child = very dangerous
    dv_young_child       = dv_history_flag == 1 & child_age < 5,

    # Large household strain
    large_household      = n_children_in_household >= 3,

    # Multi-child + young = supervision strain
    multi_child_young    = n_children_in_household >= 3 & child_age < 5,

    # Compounding risk score (0-4 scale)
    compounded_risk      = dv_history_flag +
                           shelter_involvement_flag +
                           prior_substantiated_flag +
                           as.integer(!is.na(substance_use_flag) & substance_use_flag == 1)
  ) %>%
  # Join substance flag from features table
  left_join(features %>% select(report_id, substance_use_flag,
                                 dv_history_flag, shelter_involvement_flag,
                                 prior_substantiated_flag),
            by='report_id', suffix=c('','.feat'))

# Distribution of compounded risk
vulnerability_features %>%
  count(compounded_risk.feat) %>%
  mutate(pct=round(n/sum(n)*100,1))

# DOMAIN INSIGHT:
# compounded_risk = 3 means DV history + shelter + prior substantiation
# ALL THREE together = extremely high risk profile
# The model should weight this heavily

## SECTION 5.5: NLP Features from Case Narratives

In [ ]:
# Case narratives contain rich information
# Let's extract signal from the free text

# Step 1: Tokenize narratives
narrative_tokens <- scr %>%
  select(report_id, narrative, outcome) %>%
  unnest_tokens(word, narrative) %>%
  anti_join(stop_words, by='word')  # remove common words

# Step 2: Which words predict substantiation?
word_substantiation <- narrative_tokens %>%
  mutate(is_subst = outcome == 'Substantiated') %>%
  group_by(word) %>%
  summarise(
    n          = n(),
    subst_rate = mean(is_subst)
  ) %>%
  filter(n >= 5) %>%
  arrange(desc(subst_rate))

cat('Words most associated with substantiation:\n')
print(head(word_substantiation, 15))

cat('\nWords least associated with substantiation:\n')
print(tail(word_substantiation, 10))

In [ ]:
# Step 3: TF-IDF - which words are distinctive per outcome?
narrative_tfidf <- narrative_tokens %>%
  count(outcome, word) %>%
  bind_tf_idf(word, outcome, n) %>%
  arrange(outcome, desc(tf_idf))

# Top distinctive words per outcome
narrative_tfidf %>%
  group_by(outcome) %>%
  slice_max(tf_idf, n=8) %>%
  ungroup() %>%
  mutate(word = reorder_within(word, tf_idf, outcome)) %>%
  ggplot(aes(x=word, y=tf_idf, fill=outcome)) +
  geom_col(show.legend=FALSE) +
  facet_wrap(~outcome, scales='free') +
  coord_flip() +
  scale_x_reordered() +
  labs(title='Most Distinctive Words by Outcome (TF-IDF)',
       subtitle='Words that distinguish each outcome from others',
       x=NULL, y='TF-IDF Score') +
  theme_minimal(base_size=11)

In [ ]:
# Step 4: Create risk keyword features from narratives
# These become model features alongside structured data

risk_keywords <- c('bruising','injury','abuse','intoxicated',
                   'unsupervised','overnight','alone','hungry')

safety_keywords <- c('improving','cooperative','engaged',
                     'resources','stable','enrolled')

narrative_features <- scr %>%
  select(report_id, narrative) %>%
  mutate(
    narrative_lower      = tolower(narrative),
    n_risk_keywords      = str_count(narrative_lower,
                            paste(risk_keywords, collapse='|')),
    n_safety_keywords    = str_count(narrative_lower,
                            paste(safety_keywords, collapse='|')),
    has_risk_keyword     = n_risk_keywords > 0,
    narrative_risk_score = n_risk_keywords - n_safety_keywords
  ) %>%
  select(-narrative_lower, -narrative)

narrative_features %>% summary()

# DOMAIN INSIGHT:
# NLP features from narratives often improve model performance
# because caseworkers encode their clinical judgment in the notes
# The model can learn to read the caseworker's written concern

## SECTION 5.6: Final Feature Engineering Pipeline

In [ ]:
# Combine ALL features into one clean table ready for modeling

final_features <- features %>%
  # Temporal features
  left_join(temporal_features %>%
    select(report_id, report_number, days_since_last, gap_shrinking,
           prior_subst_count, n_prior_reporters),
    by='report_id') %>%
  # Severity features
  left_join(severity_features %>%
    select(report_id, allegation_severity, severity_escalating,
           max_prior_severity, repeat_allegation_type),
    by='report_id') %>%
  # Reporter features
  left_join(reporter_features %>%
    select(report_id, distinct_reporters_so_far, is_mandated,
           is_anonymous, is_medical),
    by='report_id') %>%
  # NLP features
  left_join(narrative_features, by='report_id') %>%
  # Clean missingness
  mutate(
    has_prior_report       = prior_reports_12mo > 0,
    days_since_last        = replace_na(days_since_last, -1),
    substance_flag_missing = is.na(substance_use_flag),
    substance_use_flag     = replace_na(substance_use_flag, 0),
    contact_missing        = is.na(days_to_first_contact),
    days_to_first_contact  = replace_na(days_to_first_contact,
                               median(days_to_first_contact, na.rm=TRUE)),
    gap_shrinking          = replace_na(gap_shrinking, FALSE),
    severity_escalating    = replace_na(severity_escalating, FALSE),
    across(where(is.logical), as.integer)
  )

cat('Final feature table:', nrow(final_features), 'rows x',
    ncol(final_features), 'columns\n')
cat('Missing values:', sum(is.na(final_features)), '\n')

# Save enriched features
write_csv(final_features, 'data/acs_features_enriched.csv')
cat('Saved to data/acs_features_enriched.csv\n')